# الفصل الحادي عشر: معالجة الأخطاء (Error Handling)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-11.ipynb)

> **الوقت المتوقع:** ~9 دقيقة قراءة · **14** قسمًا · **31** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **11.1** لماذا يحتاج كل برنامج احترافي لمعالجة الأخطاء؟
- **11.2** فهم الاستثناءات (Exceptions)
- **11.3** الصيغة الأساسية: try / except
- **11.4** else و finally: إكمال الصورة
- **11.5** إطلاق الأخطاء بنفسك: raise
- **11.6** إنشاء استثناءات مخصصة (Custom Exceptions)
- **11.7** فلسفتان في البرمجة: EAFP مقابل LBYL
- **11.8** دالة assert: التحقق أثناء التطوير
- **11.9** مقدمة مختصرة لـ logging بدلاً من print
- **11.10** مشروع تطبيقي: نظام تسجيل دخول آمن
- **11.11** تمارين
- **11.12** اختبر فهمك (Quiz)
- **11.13** ملخص الفصل (Cheat Sheet)
- **11.14** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

## 11.1 لماذا يحتاج كل برنامج احترافي لمعالجة الأخطاء؟

تخيّل تطبيق بنك يطلب من المستخدم إدخال مبلغ للسحب. ماذا لو كتب المستخدم حروفاً بدلاً من أرقام؟ ماذا لو حاول السحب أكثر من رصيده؟ ماذا لو انقطع الاتصال بالإنترنت أثناء معالجة العملية؟

بالطريقة التي تعلمناها حتى الآن، أي من هذه الحالات ستجعل البرنامج **ينهار فوراً** برسالة خطأ مخيفة (Traceback) يراها المستخدم النهائي — وهذا غير مقبول إطلاقاً في أي تطبيق حقيقي.

```python
balance = 1000
withdrawal = input("أدخل مبلغ السحب: ")  # المستخدم يكتب "مئة"
balance -= int(withdrawal)  # 💥 ValueError: invalid literal for int()
# البرنامج ينهار، والمستخدم يرى رسالة تقنية غير مفهومة
```

🧾 **للقراءة فقط** — يحتاج إدخالًا يدويًا.

المبرمج المحترف لا يفترض أن كل شيء سيسير كما خطط له. إنه **يتوقع** الأخطاء المحتملة، ويتعامل معها بأناقة: رسالة واضحة للمستخدم، محاولة بديلة، أو تسجيل الخطأ للمراجعة لاحقاً — كل ذلك **دون** أن ينهار البرنامج بالكامل.

في هذا الفصل، ستتعلم فلسفة Python الكاملة في التعامل مع الأخطاء، من الأساسيات إلى إنشاء أخطائك المخصصة، وستكتسب عقلية «البرمجة الدفاعية» التي تميز الكود الجاهز للإنتاج (Production-Ready) عن كود التجارب.

## 11.2 فهم الاستثناءات (Exceptions)

### ما هو الاستثناء؟

**الاستثناء (Exception)** هو حدث غير طبيعي يحدث أثناء تنفيذ البرنامج، يُوقف التنفيذ العادي إذا لم تتم معالجته. تذكّر من الفصل الثاني أنواع الأخطاء التي واجهناها:

```python
print(10 / 0)        # ZeroDivisionError
print(int("abc"))    # ValueError
print(my_list[99])   # IndexError
print(my_dict["x"])  # KeyError
print(undefined_var) # NameError
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `my_dict`، `my_list`، `undefined_var` من مثال سابق.

كل خطأ من هذه له **نوع (Exception Type)** محدد، وهذه هي قوة نظام الاستثناءات في Python: يمكنك التعامل مع كل نوع خطأ بطريقة مختلفة.

### تشريح Traceback

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    def divide(a, b):
        return a / b

    result = divide(10, 0)
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

```
Traceback (most recent call last):
  File "script.py", line 4, in <module>
    result = divide(10, 0)
             ^^^^^^^^^^^^^
  File "script.py", line 2, in divide
    return a / b
           ~~^~~
ZeroDivisionError: division by zero
```

⚠️ **للقراءة فقط** — ناتج خطأ متوقّع.

**كيفية القراءة**: اقرأ من **الأسفل للأعلى**. السطر الأخير يخبرك بنوع الخطأ ورسالته. الأسطر الأعلى تريك «سلسلة الاستدعاءات» (Call Stack) التي أدت للخطأ — أي الدالة التي استدعت الدالة التي استدعت... إلخ.

## 11.3 الصيغة الأساسية: try / except

In [ ]:
# 📝 نسخة الدفتر: حقل نموذج بدل الإدخال اليدوي — عدّل القيمة ثم أعد التشغيل
try:
    number = 10  #@param {type:"integer"}
    print(f"ضعف الرقم: {number * 2}")
except ValueError:
    print("❌ خطأ: الرجاء إدخال رقم صحيح فقط.")

print("البرنامج استمر في العمل!")

**كيف يعمل**:
1. Python تُنفّذ الكود داخل `try`
2. إذا حدث خطأ من النوع المحدد في `except`، يتوقف تنفيذ `try` فوراً، وينتقل التنفيذ إلى كتلة `except`
3. إذا لم يحدث أي خطأ، تُتجاهل كتلة `except` بالكامل
4. البرنامج يستمر بالعمل بعد كتلة `try/except` بدلاً من الانهيار

### معالجة أنواع أخطاء متعددة

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    def safe_divide(a: float, b: float) -> float | None:
        try:
            return a / b
        except ZeroDivisionError:
            print("❌ لا يمكن القسمة على صفر.")
            return None
        except TypeError:
            print("❌ يجب أن تكون القيم أرقاماً.")
            return None

    print(safe_divide(10, 0))        # ❌ لا يمكن القسمة على صفر. → None
    print(safe_divide(10, "abc"))    # ❌ يجب أن تكون القيم أرقاماً. → None
    print(safe_divide(10, 2))        # 5.0
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

### معالجة عدة أخطاء بنفس الطريقة

```python
try:
    value = my_dict[key]
    result = 100 / value
except (KeyError, ZeroDivisionError) as e:
    print(f"حدث خطأ: {e}")
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `key`، `my_dict` من مثال سابق.

### التقاط كل الأخطاء (استخدمه بحذر شديد!)

```python
try:
    risky_operation()
except Exception as e:
    print(f"حدث خطأ غير متوقع: {e}")
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `risky_operation` من مثال سابق.

> ⚠️ **تحذير خطير جداً - أخطر عادة سيئة في معالجة الأخطاء**:

```python
# ❌ الأسوأ على الإطلاق - لا تفعل هذا أبداً!
try:
    risky_operation()
except:  # bare except - يلتقط كل شيء حتى KeyboardInterrupt!
    pass  # يُخفي الخطأ تماماً دون أي أثر!
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `risky_operation` من مثال سابق.

هذا النمط يُسمى "Silent Failure" (الفشل الصامت) — البرنامج يستمر وكأن لا شيء حدث، بينما خطأ حقيقي حدث واختفى بدون أي أثر، مما يجعل تصحيح الأخطاء لاحقاً شبه مستحيل. **القاعدة الذهبية**: التقط دائماً نوع الخطأ المحدد الذي تتوقعه، ولا تستخدم `except:` فارغة أبداً. إذا احتجت فعلاً التقاط كل شيء، استخدم `except Exception as e:` على الأقل (لتجنب التقاط إشارات النظام الحرجة)، واطبع/سجّل تفاصيل الخطأ دائماً.

### هرمية الاستثناءات (Exception Hierarchy)

جميع الأخطاء في Python ترث من فئة أساسية واحدة: `BaseException`. فهم الهرمية يساعدك في معرفة أي استثناء يُغطي أي حالات:

```
BaseException
 ├── SystemExit
 ├── KeyboardInterrupt
 └── Exception                    ← كل أخطائك المخصصة يجب أن ترث من هنا
      ├── ArithmeticError
      │    └── ZeroDivisionError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── ValueError
      ├── TypeError
      ├── NameError
      ├── AttributeError
      ├── FileNotFoundError
      ├── OSError
      └── ... والمزيد
```

> 💡 **نصيحة الخبراء**: إذا كتبت `except Exception` فستلتقط كل الأخطاء الموجودة تحتها في الهرمية، لكنها لن تلتقط `SystemExit` و`KeyboardInterrupt` (أي ضغطة Ctrl+C)، وهذا هو السلوك الصحيح والمرغوب — لا تريد منع المستخدم من إيقاف برنامجه بـ Ctrl+C!

```python
# ترتيب except مهم جداً! الأكثر تحديداً أولاً
try:
    value = my_list[10]
except IndexError:
    print("فهرس خارج النطاق")
except LookupError:  # تأتي بعد IndexError لأنها أعم منها
    print("خطأ بحث عام")
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `my_list` من مثال سابق.

> ⚠️ **خطأ شائع**: وضع `except Exception` **قبل** الأخطاء المحددة يجعل الأخطاء المحددة لا تُلتقط أبداً (لأن `Exception` تلتقطها جميعاً قبل الوصول لها):

In [ ]:
try:
    x = 1 / 0
except Exception:
    print("خطأ عام")  # هذا ما سيُطبع
except ZeroDivisionError:  # ⚠️ ميت! لن يُنفَّذ أبداً
    print("قسمة على صفر")

**القاعدة**: رتّب كتل `except` من الأكثر تحديداً إلى الأعم دائماً.

## 11.4 else و finally: إكمال الصورة

### الصيغة الكاملة

```python
try:
    # الكود الذي قد يُطلق خطأ
    pass
except SomeError:
    # يُنفَّذ فقط إذا حدث هذا الخطأ المحدد
    pass
else:
    # يُنفَّذ فقط إذا لم يحدث أي خطأ في try
    pass
finally:
    # يُنفَّذ دائماً، سواء حدث خطأ أم لا
    pass
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `SomeError` من مثال سابق.

### مثال شامل عملي

In [ ]:
def read_and_process_number(text: str) -> None:
    try:
        number = int(text)
    except ValueError:
        print(f"❌ '{text}' ليس رقماً صحيحاً.")
    else:
        # يُنفَّذ فقط إذا نجح التحويل بدون خطأ
        print(f"✅ تم تحويل الرقم بنجاح: {number}")
        print(f"   ضعفه: {number * 2}")
    finally:
        # يُنفَّذ دائماً - مثالي للتنظيف
        print("--- انتهت المحاولة ---")

read_and_process_number("42")
print()
read_and_process_number("abc")

**الناتج:**

```
✅ تم تحويل الرقم بنجاح: 42
   ضعفه: 84
--- انتهت المحاولة ---

❌ 'abc' ليس رقماً صحيحاً.
--- انتهت المحاولة ---
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

> 💡 **نصيحة الخبراء - لماذا نستخدم `else` بدلاً من وضع الكود داخل `try` مباشرة؟**

```python
# أقل وضوحاً - أي سطر بالضبط قد يُطلق الخطأ؟
try:
    number = int(text)
    print(f"ضعفه: {number * 2}")  # ماذا لو هذا السطر نفسه فشل لسبب آخر؟
except ValueError:
    print("خطأ")

# أوضح بكثير - try تحتوي فقط على الكود المحتمل فشله
try:
    number = int(text)
except ValueError:
    print("خطأ")
else:
    print(f"ضعفه: {number * 2}")  # نعلم بالتأكيد أن number تم تحويله بنجاح
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `text` من مثال سابق.

استخدام `else` يفصل بوضوح «الكود الخطير» (داخل try) عن «الكود الذي يعتمد على نجاحه» (داخل else)، مما يحسّن وضوح الكود ويمنع التقاط أخطاء غير مقصودة.

### متى تستخدم `finally`؟

`finally` مثالية لعمليات «التنظيف» التي يجب أن تحدث دائماً، بغض النظر عن نجاح أو فشل العملية: إغلاق ملف، قطع اتصال بقاعدة بيانات، تحرير موارد.

In [ ]:
def process_file(filename: str) -> None:
    file = None
    try:
        file = open(filename, "r", encoding="utf-8")
        content = file.read()
        print(f"تم قراءة {len(content)} حرف.")
    except FileNotFoundError:
        print(f"❌ الملف '{filename}' غير موجود.")
    finally:
        if file is not None:
            file.close()
            print("✅ تم إغلاق الملف.")  # يحدث دائماً!

process_file("not_exist.txt")

> 💡 **نصيحة الخبراء**: في الفصل السادس عشر سنتعلم `with open(...) as file:` التي تُغني تماماً عن الحاجة لـ `finally` اليدوية لإغلاق الملفات — إنها تفعل ذلك تلقائياً باستخدام **Context Managers**. لكن فهم `finally` ضروري لفهم أي موارد أخرى تحتاج تنظيفاً يدوياً (اتصالات شبكة، قواعد بيانات، أقفال Threading).

## 11.5 إطلاق الأخطاء بنفسك: raise

أحياناً تريد أنت أن «تُطلق» خطأً عمداً عندما تكتشف حالة غير منطقية، حتى لو لم تُطلقها Python تلقائياً.

In [ ]:
def withdraw(balance: float, amount: float) -> float:
    if amount <= 0:
        raise ValueError("مبلغ السحب يجب أن يكون موجباً.")
    if amount > balance:
        raise ValueError("الرصيد غير كافٍ لإتمام العملية.")
    return balance - amount

try:
    new_balance = withdraw(1000, 1500)
except ValueError as e:
    print(f"❌ فشلت العملية: {e}")

**الناتج:** `❌ فشلت العملية: الرصيد غير كافٍ لإتمام العملية.`

### إعادة إطلاق الخطأ (Re-raising)

أحياناً تريد التقاط خطأ لتسجيله، ثم إطلاقه من جديد ليتعامل معه مستوى أعلى من الكود:

In [ ]:
def process_data(data: list) -> float:
    try:
        return sum(data) / len(data)
    except ZeroDivisionError:
        print("⚠️ تحذير: القائمة فارغة، سيتم إعادة إطلاق الخطأ.")
        raise  # raise بدون معامل = إعادة إطلاق نفس الخطأ الملتقط

try:
    process_data([])
except ZeroDivisionError:
    print("❌ المستوى الأعلى: تم التعامل مع القائمة الفارغة.")

### تسلسل الاستثناءات (Exception Chaining)

In [ ]:
def parse_age(text: str) -> int:
    try:
        return int(text)
    except ValueError as e:
        raise ValueError(f"فشل تحليل العمر من النص: '{text}'") from e

try:
    parse_age("twenty")
except ValueError as e:
    print(e)
    print(f"السبب الأصلي: {e.__cause__}")

**الناتج:**

```
فشل تحليل العمر من النص: 'twenty'
السبب الأصلي: invalid literal for int() with base 10: 'twenty'
```

🧾 **للقراءة فقط** — مقتطف غير مكتمل.

> 💡 **نصيحة الخبراء**: استخدام `raise NewError(...) from original_error` يحافظ على «السلسلة الكاملة» للخطأ الأصلي في الـ Traceback، مما يسهّل تصحيح الأخطاء كثيراً بدلاً من إخفاء السبب الجذري.

## 11.6 إنشاء استثناءات مخصصة (Custom Exceptions)

في المشاريع الحقيقية، الأخطاء المدمجة في Python (`ValueError`, `TypeError`...) لا تكفي دائماً للتعبير عن أخطاء خاصة بمجال عملك (Business Logic).

In [ ]:
class InsufficientFundsError(Exception):
    """تُطلق عندما يحاول المستخدم سحب مبلغ أكبر من رصيده."""
    pass

class InvalidAmountError(Exception):
    """تُطلق عندما يكون المبلغ المُدخل غير منطقي (سالب أو صفر)."""
    pass

def withdraw(balance: float, amount: float) -> float:
    if amount <= 0:
        raise InvalidAmountError(f"المبلغ {amount} غير صالح؛ يجب أن يكون موجباً.")
    if amount > balance:
        raise InsufficientFundsError(
            f"طلبت سحب {amount}، لكن رصيدك الحالي {balance} فقط."
        )
    return balance - amount

try:
    withdraw(500, 1000)
except InsufficientFundsError as e:
    print(f"💰 خطأ في الرصيد: {e}")
except InvalidAmountError as e:
    print(f"🔢 خطأ في المبلغ: {e}")

**الناتج:** `💰 خطأ في الرصيد: طلبت سحب 1000، لكن رصيدك الحالي 500 فقط.`

> 💡 **لماذا نُنشئ استثناءات مخصصة؟**
> 1. **وضوح أكبر**: `InsufficientFundsError` أوضح بكثير من `ValueError` عامة
> 2. **معالجة دقيقة**: يمكنك التعامل مع كل حالة عمل (Business Case) بطريقة مختلفة تماماً
> 3. **توثيق ذاتي**: الكود الذي يقرأ `except InsufficientFundsError` يفهم فوراً السيناريو دون شرح إضافي

### هرمية استثناءات مخصصة

يمكنك بناء هرمية من الأخطاء المخصصة الخاصة بتطبيقك، تماماً كما تفعل Python داخلياً:

In [ ]:
class BankError(Exception):
    """الفئة الأساسية لكل أخطاء النظام البنكي."""
    pass

class InsufficientFundsError(BankError):
    """رصيد غير كافٍ."""
    pass

class InvalidAmountError(BankError):
    """مبلغ غير صالح."""
    pass

class AccountFrozenError(BankError):
    """الحساب مجمّد."""
    pass

def process_transaction(account_status: str, balance: float, amount: float) -> float:
    if account_status == "frozen":
        raise AccountFrozenError("هذا الحساب مجمّد حالياً.")
    if amount <= 0:
        raise InvalidAmountError("المبلغ يجب أن يكون موجباً.")
    if amount > balance:
        raise InsufficientFundsError("الرصيد غير كافٍ.")
    return balance - amount

try:
    process_transaction("frozen", 1000, 100)
except BankError as e:  # التقاط أي خطأ بنكي بغض النظر عن نوعه الدقيق
    print(f"🏦 خطأ في النظام البنكي: {type(e).__name__}: {e}")

**الناتج:** `🏦 خطأ في النظام البنكي: AccountFrozenError: هذا الحساب مجمّد حالياً.`

### إضافة بيانات إضافية للاستثناء المخصص

In [ ]:
class ValidationError(Exception):
    """خطأ تحقق من صحة البيانات، مع تفاصيل الحقل المُخالف."""

    def __init__(self, field: str, message: str):
        self.field = field
        self.message = message
        super().__init__(f"خطأ في حقل '{field}': {message}")

def validate_age(age: int) -> None:
    if age < 0 or age > 150:
        raise ValidationError(field="age", message="يجب أن يكون بين 0 و150")

try:
    validate_age(-5)
except ValidationError as e:
    print(f"الحقل المُخالف: {e.field}")
    print(f"الرسالة: {e.message}")

> 🔄 **مقارنة مع لغات أخرى**: نظام الاستثناءات في Python مشابه جداً لـ Java وC#، حيث توجد هرمية استثناءات، وإمكانية إنشاء استثناءات مخصصة ترث من فئة أساسية. أما في لغات مثل C أو Go، لا يوجد نظام استثناءات مدمج بنفس القوة؛ يعتمد المبرمجون على إرجاع قيم خطأ (Error Codes) أو أزواج `(result, error)` بشكل يدوي.

## 11.7 فلسفتان في البرمجة: EAFP مقابل LBYL

هذا مفهوم فلسفي عميق يميز أسلوب Python عن لغات أخرى.

### LBYL - Look Before You Leap (انظر قبل أن تقفز)

In [ ]:
# تحقق أولاً، ثم نفّذ
import os

if os.path.exists("data.txt"):
    with open("data.txt") as f:
        content = f.read()
else:
    print("الملف غير موجود")

### EAFP - Easier to Ask Forgiveness than Permission (أسهل أن تطلب الصفح من أن تطلب الإذن)

In [ ]:
# نفّذ مباشرة، وتعامل مع الخطأ إن حدث
try:
    with open("data.txt") as f:
        content = f.read()
except FileNotFoundError:
    print("الملف غير موجود")

> 💡 **أيهما يُفضّل في Python؟** الأسلوب **الثاني (EAFP)** هو الأسلوب «الـPythonic» الموصى به رسمياً. لماذا؟
> 1. **تجنب Race Conditions**: بين لحظة `os.path.exists()` ولحظة `open()`، قد يُحذف الملف فعلياً من قبل عملية أخرى! بينما `try/except` يتعامل مع اللحظة الفعلية للتنفيذ.
> 2. **أداء أفضل عادةً**: التحقق المسبق يعني فحصاً إضافياً دائماً، بينما `try/except` لا يكلّف شيئاً إضافياً إذا لم يحدث خطأ فعلياً.
> 3. **كود أنظف**: تقل الحاجة لسلاسل `if` متداخلة للتحقق من كل شرط ممكن مسبقاً.

> ⚠️ **تنبيه**: هذا لا يعني أن `if` للتحقق المسبق سيء دائماً! في حالات بسيطة وواضحة (مثل `if value is not None:`)، التحقق المباشر أوضح من `try/except`. القاعدة العملية: استخدم EAFP عندما يكون الفشل **محتملاً لكن نادراً** وستتعامل معه برسالة/بديل، واستخدم LBYL عندما يكون التحقق **بسيطاً جداً وسريعاً** (مثل فحص قيمة None).

## 11.8 دالة assert: التحقق أثناء التطوير

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    def calculate_average(numbers: list[float]) -> float:
        assert len(numbers) > 0, "القائمة يجب ألا تكون فارغة"
        return sum(numbers) / len(numbers)

    print(calculate_average([10, 20, 30]))  # 20.0
    # print(calculate_average([]))  
    # 💥 AssertionError: القائمة يجب ألا تكون فارغة
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

> ⚠️ **تحذير حرج**: `assert` **ليست** بديلاً عن `try/except` لمعالجة أخطاء المستخدم أو البيانات الخارجية! السبب: عند تشغيل Python بوضع «التحسين» (`python -O script.py`)، تُحذف كل عبارات `assert` بالكامل من الكود المُنفَّذ! لذلك:
> - استخدم `assert` فقط للتحقق من افتراضات داخلية أثناء **التطوير والاختبار** (مثل: «هذه الدالة يجب ألا تُستدعى أبداً بقائمة فارغة، وإذا حدث فهذا خطأ برمجي مني أنا المطور»)
> - استخدم `raise` و`try/except` دائماً للتعامل مع أي بيانات قد تأتي من **مستخدم حقيقي أو مصدر خارجي** (إدخال، ملف، API، قاعدة بيانات)

```python
# ❌ خطأ: استخدام assert للتحقق من مدخلات مستخدم
def set_age(age: int) -> None:
    assert age >= 0, "العمر يجب أن يكون موجباً"  # قد يُحذف بالكامل!

# ✅ صحيح: استخدام raise للتحقق من مدخلات مستخدم
def set_age(age: int) -> None:
    if age < 0:
        raise ValueError("العمر يجب أن يكون موجباً")
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

## 11.9 مقدمة مختصرة لـ logging بدلاً من print

في المشاريع الحقيقية، لا نستخدم `print()` لتسجيل الأخطاء؛ نستخدم وحدة `logging` المدمجة التي تمنحنا مستويات خطورة، وتوقيت تلقائي، وإمكانية الحفظ بملفات.

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    import logging

    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

    def divide(a: float, b: float) -> float | None:
        try:
            return a / b
        except ZeroDivisionError:
            logging.error(f"محاولة قسمة {a} على صفر!")
            return None

    divide(10, 0)
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

**الناتج:**

```
2025-01-15 10:30:45,123 - ERROR - محاولة قسمة 10 على صفر!
```

> 💡 **نصيحة الخبراء**: `logging` خارج نطاق هذا الفصل بالتفصيل (سنتعمق فيه لاحقاً عند الحديث عن بناء تطبيقات حقيقية)، لكن تذكّر هذه القاعدة من الآن: **print للتجربة السريعة، logging للمشاريع الحقيقية**. الفرق الجوهري أن `logging` يمنحك تحكماً كاملاً (تشغيل/إيقاف أنواع معينة من الرسائل، حفظها بملف، إرسالها لنظام مراقبة) دون تعديل الكود نفسه.

## 11.10 مشروع تطبيقي: نظام تسجيل دخول آمن

نظام متكامل يستخدم كل ما تعلمناه: استثناءات مخصصة، هرمية أخطاء، try/except/else/finally، وEAFP.

In [ ]:
class AuthError(Exception):
    """الفئة الأساسية لكل أخطاء المصادقة."""
    pass

class UserNotFoundError(AuthError):
    """المستخدم غير موجود في النظام."""
    pass

class IncorrectPasswordError(AuthError):
    """كلمة المرور غير صحيحة."""
    pass

class AccountLockedError(AuthError):
    """الحساب مقفل بسبب محاولات فاشلة متكررة."""
    pass

class WeakPasswordError(AuthError):
    """كلمة مرور ضعيفة عند التسجيل."""
    pass

# قاعدة بيانات وهمية (في الواقع ستكون قاعدة بيانات حقيقية)
users_db: dict[str, dict] = {
    "ahmed@test.com": {"password": "Secure123", "failed_attempts": 0, "locked": False},
}

MAX_FAILED_ATTEMPTS = 3

def register(email: str, password: str) -> None:
    """تسجيل مستخدم جديد مع التحقق من قوة كلمة المرور."""
    if len(password) < 8:
        raise WeakPasswordError("كلمة المرور يجب أن تكون 8 أحرف على الأقل.")
    if not any(char.isdigit() for char in password):
        raise WeakPasswordError("كلمة المرور يجب أن تحتوي رقماً واحداً على الأقل.")

    if email in users_db:
        raise ValueError("هذا البريد الإلكتروني مسجّل مسبقاً.")

    users_db[email] = {"password": password, "failed_attempts": 0, "locked": False}
    print(f"✅ تم تسجيل المستخدم {email} بنجاح.")

def login(email: str, password: str) -> bool:
    """محاولة تسجيل دخول، مع حماية من المحاولات المتكررة."""
    try:
        user = users_db[email]
    except KeyError:
        raise UserNotFoundError(f"لا يوجد مستخدم بالبريد: {email}")

    if user["locked"]:
        raise AccountLockedError("هذا الحساب مقفل. تواصل مع الدعم الفني.")

    if user["password"] != password:
        user["failed_attempts"] += 1
        if user["failed_attempts"] >= MAX_FAILED_ATTEMPTS:
            user["locked"] = True
            raise AccountLockedError(
                "تم قفل الحساب بسبب تجاوز عدد المحاولات الفاشلة المسموحة."
            )
        remaining = MAX_FAILED_ATTEMPTS - user["failed_attempts"]
        raise IncorrectPasswordError(
            f"كلمة المرور خاطئة. المحاولات المتبقية: {remaining}"
        )

    user["failed_attempts"] = 0  # إعادة تصفير عند النجاح
    return True

def run_login_attempt(email: str, password: str) -> None:
    """واجهة تفاعلية تلتقط كل أنواع أخطاء المصادقة بوضوح."""
    try:
        login(email, password)
    except UserNotFoundError as e:
        print(f"👤 {e}")
    except AccountLockedError as e:
        print(f"🔒 {e}")
    except IncorrectPasswordError as e:
        print(f"🔑 {e}")
    except AuthError as e:
        print(f"⚠️ خطأ مصادقة غير متوقع: {e}")
    else:
        print(f"✅ تم تسجيل الدخول بنجاح! أهلاً بك {email}")
    finally:
        print("-" * 40)

# --- سيناريوهات تجريبية ---
if __name__ == "__main__":
    try:
        register("sara@test.com", "weak")
    except WeakPasswordError as e:
        print(f"❌ فشل التسجيل: {e}")
    print("-" * 40)

    register("sara@test.com", "Strong123")
    print("-" * 40)

    run_login_attempt("unknown@test.com", "anything")   # UserNotFoundError
    run_login_attempt("ahmed@test.com", "wrong_pass")   # IncorrectPasswordError (محاولة 1)
    run_login_attempt("ahmed@test.com", "wrong_pass")   # IncorrectPasswordError (محاولة 2)
    run_login_attempt("ahmed@test.com", "wrong_pass")   # AccountLockedError (محاولة 3 - قفل!)
    run_login_attempt("ahmed@test.com", "Secure123")    # AccountLockedError (مقفل مسبقاً)
    run_login_attempt("sara@test.com", "Strong123")     # ✅ نجاح

**الناتج المتوقع:**

```
❌ فشل التسجيل: كلمة المرور يجب أن تكون 8 أحرف على الأقل.
----------------------------------------
✅ تم تسجيل المستخدم sara@test.com بنجاح.
----------------------------------------
👤 لا يوجد مستخدم بالبريد: unknown@test.com
----------------------------------------
🔑 كلمة المرور خاطئة. المحاولات المتبقية: 2
----------------------------------------
🔑 كلمة المرور خاطئة. المحاولات المتبقية: 1
----------------------------------------
🔒 تم قفل الحساب بسبب تجاوز عدد المحاولات الفاشلة المسموحة.
----------------------------------------
🔒 هذا الحساب مقفل. تواصل مع الدعم الفني.
----------------------------------------
✅ تم تسجيل الدخول بنجاح! أهلاً بك sara@test.com
----------------------------------------
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

<details>
<summary>✅ تحدي إضافي: دالة إعادة تعيين كلمة المرور (الحل)</summary>

In [ ]:
def reset_password(email: str, new_password: str) -> None:
    """إعادة تعيين كلمة مرور، مع إلغاء القفل إن وُجد."""
    try:
        user = users_db[email]
    except KeyError:
        raise UserNotFoundError(f"لا يوجد مستخدم بالبريد: {email}")

    if len(new_password) < 8 or not any(c.isdigit() for c in new_password):
        raise WeakPasswordError("كلمة المرور الجديدة ضعيفة جداً.")

    user["password"] = new_password
    user["failed_attempts"] = 0
    user["locked"] = False
    print(f"🔓 تم إعادة تعيين كلمة المرور وفك قفل الحساب لـ {email}")

try:
    reset_password("ahmed@test.com", "NewPass123")
except AuthError as e:
    print(f"❌ {e}")

</details>

## 11.11 تمارين

### 🟢 مستوى مبتدئ

**تمرين 1**: اكتب كود `try/except` يطلب من المستخدم رقماً، ويطبع «رقم صحيح» أو «خطأ: ليس رقماً» حسب الحالة.

**تمرين 2**: اكتب دالة `safe_list_access(lst: list, index: int)` تُرجع العنصر في الفهرس المطلوب، أو `None` إذا كان الفهرس خارج النطاق (باستخدام `IndexError`).

<details>
<summary>✅ الحل</summary>

In [ ]:
# 📝 نسخة الدفتر: حقل نموذج بدل الإدخال اليدوي — عدّل القيمة ثم أعد التشغيل
# تمرين 1
try:
    num = 10  #@param {type:"integer"}
    print("رقم صحيح")
except ValueError:
    print("خطأ: ليس رقماً")

# تمرين 2
def safe_list_access(lst: list, index: int):
    try:
        return lst[index]
    except IndexError:
        return None

print(safe_list_access([1, 2, 3], 1))   # 2
print(safe_list_access([1, 2, 3], 10))  # None

</details>

### 🟡 مستوى متوسط

**تمرين 3**: اكتب استثناءً مخصصاً `NegativeValueError`، ودالة `calculate_square_root(n)` تُطلقه إذا كان الرقم سالباً، وتُرجع الجذر التربيعي خلاف ذلك.

**تمرين 4**: اكتب دالة `parse_config(data: dict)` تقرأ المفاتيح `"host"` و`"port"` من قاموس إعدادات، وتُطلق استثناءً مخصصاً `MissingConfigError` مع اسم المفتاح المفقود إن لم يكن موجوداً.

<details>
<summary>✅ الحل</summary>

In [ ]:
import math

# تمرين 3
class NegativeValueError(Exception):
    pass

def calculate_square_root(n: float) -> float:
    if n < 0:
        raise NegativeValueError(f"لا يمكن حساب جذر رقم سالب: {n}")
    return math.sqrt(n)

try:
    print(calculate_square_root(-9))
except NegativeValueError as e:
    print(f"❌ {e}")

# تمرين 4
class MissingConfigError(Exception):
    def __init__(self, key: str):
        self.key = key
        super().__init__(f"الإعداد المطلوب '{key}' غير موجود.")

def parse_config(data: dict) -> tuple[str, int]:
    try:
        host = data["host"]
    except KeyError:
        raise MissingConfigError("host")

    try:
        port = data["port"]
    except KeyError:
        raise MissingConfigError("port")

    return host, port

try:
    parse_config({"host": "localhost"})
except MissingConfigError as e:
    print(f"❌ {e}")  # ❌ الإعداد المطلوب 'port' غير موجود.

</details>

### 🔴 تحدي

**تمرين 5**: اكتب نظام معالجة طلبات API وهمي: دالة `process_api_request(endpoint: str, data: dict)` تتحقق من:
- أن `endpoint` موجود في قائمة مسارات مسموحة، وإلا تُطلق `EndpointNotFoundError`
- أن `data` تحتوي حقل `"user_id"`، وإلا تُطلق `MissingFieldError`
- أن `user_id` رقم صحيح موجب، وإلا تُطلق `InvalidFieldError`

استخدم هرمية استثناءات (`APIError` كفئة أساسية)، واكتب دالة تستدعي `process_api_request` وتلتقط كل الحالات بوضوح.

<details>
<summary>✅ الحل</summary>

In [ ]:
class APIError(Exception):
    """الفئة الأساسية لكل أخطاء API."""
    pass

class EndpointNotFoundError(APIError):
    def __init__(self, endpoint: str):
        self.endpoint = endpoint
        super().__init__(f"المسار '{endpoint}' غير موجود.")

class MissingFieldError(APIError):
    def __init__(self, field: str):
        self.field = field
        super().__init__(f"الحقل المطلوب '{field}' مفقود.")

class InvalidFieldError(APIError):
    def __init__(self, field: str, reason: str):
        self.field = field
        self.reason = reason
        super().__init__(f"الحقل '{field}' غير صالح: {reason}")

VALID_ENDPOINTS = {"/users", "/orders", "/products"}

def process_api_request(endpoint: str, data: dict) -> dict:
    if endpoint not in VALID_ENDPOINTS:
        raise EndpointNotFoundError(endpoint)

    if "user_id" not in data:
        raise MissingFieldError("user_id")

    user_id = data["user_id"]
    if not isinstance(user_id, int) or user_id <= 0:
        raise InvalidFieldError("user_id", "يجب أن يكون رقماً صحيحاً موجباً")

    return {"status": "success", "endpoint": endpoint, "user_id": user_id}

def handle_request(endpoint: str, data: dict) -> None:
    try:
        result = process_api_request(endpoint, data)
    except EndpointNotFoundError as e:
        print(f"404 - {e}")
    except MissingFieldError as e:
        print(f"400 - حقل مفقود: {e}")
    except InvalidFieldError as e:
        print(f"400 - حقل غير صالح: {e}")
    except APIError as e:
        print(f"500 - خطأ API غير متوقع: {e}")
    else:
        print(f"200 - {result}")

handle_request("/comments", {"user_id": 1})         # 404
handle_request("/users", {})                         # 400 مفقود
handle_request("/users", {"user_id": -5})            # 400 غير صالح
handle_request("/users", {"user_id": 42})            # 200 نجاح

</details>

## 11.12 اختبر فهمك (Quiz)

<details>
<summary>❓ السؤال 1: ما الفرق بين else و finally في try/except؟</summary>

**الإجابة**: كتلة `else` تُنفَّذ فقط إذا **لم يحدث أي خطأ** في `try`. كتلة `finally` تُنفَّذ **دائماً**، سواء حدث خطأ أم لا، وحتى لو كان هناك `return` داخل `try` أو `except`. `finally` مخصصة لعمليات التنظيف الحتمية (إغلاق ملفات، اتصالات).
</details>

<details>
<summary>❓ السؤال 2: لماذا تُعتبر except: الفارغة (bare except) ممارسة خطيرة؟</summary>

**الإجابة**: لأنها تلتقط **كل** أنواع الاستثناءات دون تمييز، بما فيها `KeyboardInterrupt` و`SystemExit` التي يجب أن تمر دون إعاقة. كما أنها تُخفي الأخطاء الحقيقية دون رسالة واضحة، مما يجعل تصحيح الأخطاء لاحقاً صعباً جداً (Silent Failure). الأفضل دائماً تحديد نوع الخطأ المتوقع بدقة.
</details>

<details>
<summary>❓ السؤال 3: متى يجب أن تستخدم استثناءً مخصصاً بدلاً من ValueError عامة؟</summary>

**الإجابة**: عندما يكون الخطأ مرتبطاً بمنطق عمل خاص بتطبيقك (Business Logic)، مثل `InsufficientFundsError` في نظام بنكي. هذا يجعل الكود أوضح للقراءة، ويسمح بمعالجة دقيقة لكل سيناريو على حدة، ويوثق نية الكود ذاتياً دون الحاجة لتعليقات إضافية.
</details>

<details>
<summary>❓ السؤال 4: ما الفرق بين EAFP و LBYL، ولماذا تُفضّل Python عادة EAFP؟</summary>

**الإجابة**: LBYL (Look Before You Leap) يعني التحقق من الشرط قبل التنفيذ (مثل `if os.path.exists(f):`)، بينما EAFP (Easier to Ask Forgiveness than Permission) يعني التنفيذ المباشر مع معالجة الخطأ إن حدث (`try/except`). Python تُفضّل EAFP لأنه يتجنب حالات Race Condition، وغالباً أداؤه أفضل عندما يكون الفشل نادراً، وينتج كوداً أنظف بدون تحققات متداخلة.
</details>

<details>
<summary>❓ السؤال 5: لماذا لا يجب استخدام assert للتحقق من مدخلات المستخدم؟</summary>

**الإجابة**: لأن عبارات `assert` تُحذف بالكامل من الكود عند تشغيل Python بوضع التحسين (`python -O`)، مما يعني أن أي تحقق مبني عليها قد يختفي تماماً في بيئة الإنتاج، تاركاً الباب مفتوحاً لبيانات غير صالحة. `assert` مخصصة فقط للتحقق من افتراضات داخلية أثناء التطوير، وليس لمعالجة أخطاء قادمة من مصادر خارجية.
</details>

## 11.13 ملخص الفصل (Cheat Sheet)

```python
# الصيغة الكاملة
try:
    risky_code()
except SpecificError as e:
    handle_specific_case(e)
except (ErrorA, ErrorB) as e:
    handle_multiple_cases(e)
except Exception as e:      # الأعم - دائماً في الأخير
    handle_general_case(e)
else:
    runs_if_no_error()
finally:
    always_runs()            # تنظيف الموارد

# إطلاق خطأ
raise ValueError("رسالة واضحة")

# إعادة إطلاق
except SomeError:
    raise  # بدون معامل

# تسلسل الاستثناءات
raise NewError("...") from original_error

# استثناء مخصص
class MyAppError(Exception):
    pass

class SpecificError(MyAppError):
    def __init__(self, detail: str):
        self.detail = detail
        super().__init__(f"فشل بسبب: {detail}")

# assert - للتطوير فقط، ليس لمدخلات المستخدم
assert condition, "رسالة إذا فشل الشرط"
```

🧾 **للقراءة فقط** — مقتطف غير مكتمل.

| الموقف | الحل الموصى به |
|---|---|
| خطأ متوقع من مستخدم/ملف/شبكة | `try/except` + استثناء محدد |
| التحقق من افتراض داخلي أثناء التطوير | `assert` |
| منطق عمل خاص بتطبيقك | استثناء مخصص يرث من `Exception` |
| تنظيف موارد (ملفات، اتصالات) | `finally` أو أفضل: `with` (الفصل 16) |
| تسجيل الأخطاء في مشروع حقيقي | `logging` بدلاً من `print` |
| خطأ غير متوقع تماماً | `except Exception as e:` (نادراً، وبحذر) |
| لا تفعل أبداً | `except:` فارغة، أو `except Exception: pass` |

## 11.14 ماذا بعد؟

الآن برامجك لا تنهار فجأة أمام مستخدميك — بل تتعامل مع المواقف الاستثنائية بأناقة واحترافية. هذه مهارة جوهرية تفصل الكود «الذي يعمل في حالات مثالية» عن الكود «الجاهز للعالم الحقيقي».

لكن حتى الآن، كل دوالنا كانت «عادية» نسبياً. في Python توجد أدوات أكثر قوة وأناقة: دوال تُكتب في سطر واحد (Lambda)، دوال تُعدّل سلوك دوال أخرى (Decorators)، دوال «تتذكر» حالتها بين الاستدعاءات (Generators و Closures). هذه الأدوات هي ما يميز الكود البايثوني (Pythonic) الاحترافي، وتُستخدم بكثافة في كل مكتبات Python الكبرى مثل Flask وFastAPI وPytest.

في **الفصل الثاني عشر: الدوال المتقدمة**، سنفتح هذا الباب بالكامل، بعمق وبأمثلة عملية تطبيقية.

---

### الخطوة التالية

- [الفصل الثاني عشر: الدوال المتقدمة (Lambda, Decorators, Generators, Closures)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-12.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل الحادي عشر — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*